# DA 6021 DLCV Project — Wave-CCFF RT-DETR-R18 Runner (2× T4 Kaggle)

**Assigned Paper:** *DETRs Beat YOLOs on Real-time Object Detection* (Zhao et al., CVPR 2024)  
**Proposed Novelty:** **Wave-CCFF (`WaveHybridEncoder`)** — Orthonormal 2D Haar Wavelet High-Frequency Subband Upsampling & Lossless Wavelet-Residual Downsampling for Small-Object Real-Time DETR.

### Quick-Start on Kaggle (Training)
1. **Right Sidebar → Session options:** `Accelerator: GPU T4 x2` | `Internet: On`
2. Click **Save Version → Save & Run All (Commit)** *(runs up to Kaggle's 12-hour session limit)* or **Run All**.
3. **Full Epoch Completion:** Both **Baseline (`RT-DETR-R18`)** and **Ours (`Wave-CCFF RT-DETR-R18`)** run **13 epochs** (`batch_size=32` across 2× T4, LR drop at epoch 11).
4. Download from **Output (`/kaggle/working/`)**:
   - `Overleaf_Upload_Wave_CCFF.zip` (`main.tex` pre-filled with your exact results)
   - `da6021_report_results_pack.zip` (all JSONs, CSVs, logs, curves, visualizations)
   - `submission_artifacts/baseline_best.pth` & `submission_artifacts/wave_ccff_best.pth`


## Step 1: Hardware Verification & Runtime Setup


In [1]:
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

NOTEBOOK_START_TIME = time.time()
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_out").resolve()
REPO_DIR = WORK_ROOT / "wave_rtdetr_repo"
ARTIFACTS_DIR = WORK_ROOT / "submission_artifacts"
TMP_ROOT = Path("/tmp/da6021_wave_rtdetr")

for d in [WORK_ROOT, REPO_DIR, ARTIFACTS_DIR, TMP_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

try:
    import pycocotools
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pycocotools>=2.0.7"])
    import pycocotools

import torch
import torchvision

NUM_GPUS = torch.cuda.device_count()
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)] if NUM_GPUS > 0 else ["CPU"]
VRAM_GB = [round(torch.cuda.get_device_properties(i).total_memory / (1024**3), 2) for i in range(NUM_GPUS)]

hw_info = {
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_version": torch.version.cuda,
    "num_gpus": NUM_GPUS,
    "gpu_names": GPU_NAMES,
    "vram_gb": VRAM_GB,
    "cpu_count": os.cpu_count(),
}
with (ARTIFACTS_DIR / "hardware_specs.json").open("w", encoding="utf-8") as f:
    json.dump(hw_info, f, indent=2)

print("Hardware & Runtime Verified:")
print(json.dumps(hw_info, indent=2))
assert NUM_GPUS >= 1, "No CUDA GPU detected! Please enable Accelerator -> GPU T4 x2 in Kaggle Session Options."

Hardware & Runtime Verified:
{
  "python": "3.13.16",
  "pytorch": "2.11.0+cu130",
  "torchvision": "0.26.0+cu130",
  "cuda_version": "13.0",
  "num_gpus": 2,
  "gpu_names": [
    "Tesla T4",
    "Tesla T4"
  ],
  "vram_gb": [
    14.56,
    14.56
  ],
  "cpu_count": 4
}


## Step 2: Unpack Wave-CCFF RT-DETR Repository & Verify Parseval / Haar Math


In [2]:
import base64
import io
import shutil
import subprocess
import sys
import tarfile

REPO_TAR_B64 = "..."

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
REPO_DIR.mkdir(parents=True, exist_ok=True)

raw_bytes = base64.b64decode(REPO_TAR_B64.encode("ascii"))
with tarfile.open(fileobj=io.BytesIO(raw_bytes), mode="r:gz") as tf:
    if sys.version_info >= (3, 12):
        tf.extractall(REPO_DIR, filter="data")
    else:
        tf.extractall(REPO_DIR)

print(f"Unpacked complete Wave-CCFF RT-DETR repository to: {REPO_DIR}")
verify_out = subprocess.check_output(
    [sys.executable, "tools/verify_wave_ccff.py"],
    cwd=str(REPO_DIR),
    text=True,
)
print(verify_out)

json_end = verify_out.rfind("}") + 1
with (ARTIFACTS_DIR / "math_verification.json").open("w", encoding="utf-8") as f:
    f.write(verify_out[:json_end])

Unpacked complete Wave-CCFF RT-DETR repository to: /kaggle/working/wave_rtdetr_repo


I0000 00:00:1795189924.933053      51 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


{
  "parseval_relative_error": 1.420753768171154e-16,
  "idwt_max_reconstruction_error": 8.881784197001252e-16,
  "nearest_vs_degenerate_haar_diff": 0.0,
  "baseline_params_M": 20.0946,
  "wave_ccff_params_M": 20.3698,
  "delta_params_M": 0.2752,
  "pretrained_keys_matched": "533/533",
  "optimizer_groups_wave": 6,
  "wave_tensors_verified_nonzero_grad": 32,
  "sample_train_loss": 18.0878,
  "deploy_output_boxes_shape": [
    2,
    300,
    4
  ]
}
ALL WAVE-CCFF VERIFICATION CHECKS PASSED.



## Step 3: Download Pretrained `RT-DETR-R18` Weights & Prepare VisDrone2019-DET in `/tmp`


In [3]:
import json
import subprocess
import sys
import time
import urllib.request

PRETRAINED_URL = "https://github.com/lyuwenyu/storage/releases/download/v0.1/rtdetr_r18vd_dec3_6x_coco_from_paddle.pth"
PRETRAINED_PATH = TMP_ROOT / "pretrained" / "rtdetr_r18vd_coco.pth"
VISDRONE_ROOT = TMP_ROOT / "visdrone"

PRETRAINED_PATH.parent.mkdir(parents=True, exist_ok=True)
if not PRETRAINED_PATH.exists() or PRETRAINED_PATH.stat().st_size < 50_000_000:
    for attempt in range(1, 4):
        try:
            print(f"[Attempt {attempt}] Downloading official COCO-pretrained RT-DETR-R18 weights ...")
            urllib.request.urlretrieve(PRETRAINED_URL, PRETRAINED_PATH)
            break
        except Exception as exc:
            print(f"Download warning: {exc}")
            if attempt == 3:
                raise
            time.sleep(3)
print(f"Pretrained checkpoint ready: {PRETRAINED_PATH} ({PRETRAINED_PATH.stat().st_size / 1e6:.1f} MB)")

cmd_prep = [
    sys.executable,
    "tools/prepare_visdrone.py",
    "--data-root", str(VISDRONE_ROOT),
    "--search-input", "/kaggle/input",
]
subprocess.run(cmd_prep, cwd=str(REPO_DIR), check=True)

with (VISDRONE_ROOT / "dataset_stats.json").open("r", encoding="utf-8") as f:
    ds_stats = json.load(f)
with (ARTIFACTS_DIR / "dataset_stats.json").open("w", encoding="utf-8") as f:
    json.dump(ds_stats, f, indent=2)
print("Dataset summary:", json.dumps(ds_stats, indent=2))

[Attempt 1] Downloading official COCO-pretrained RT-DETR-R18 weights ...
Pretrained checkpoint ready: /tmp/da6021_wave_rtdetr/pretrained/rtdetr_r18vd_coco.pth (81.0 MB)
[VisDrone] Downloading https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip -> /tmp/da6021_wave_rtdetr/visdrone/VisDrone2019-DET-train.zip ...
[VisDrone] Extracting /tmp/da6021_wave_rtdetr/visdrone/VisDrone2019-DET-train.zip ...
[VisDrone] Converted {'split': 'VisDrone2019-DET-train', 'num_images': 6471, 'num_annotations': 343197, 'small_obj_ratio': 0.6047, 'out_json': '/tmp/da6021_wave_rtdetr/visdrone/annotations/instances_train.json'}
[VisDrone] Downloading https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip -> /tmp/da6021_wave_rtdetr/visdrone/VisDrone2019-DET-val.zip ...
[VisDrone] Extracting /tmp/da6021_wave_rtdetr/visdrone/VisDrone2019-DET-val.zip ...
[VisDrone] Converted {'split': 'VisDrone2019-DET-val', 'num_images': 548, 'num_annotations': 

## Step 4: Multi-GPU (`2× T4` DDP) Training Runner


In [4]:
import json
import os
import subprocess
import sys
import time


def elapsed_minutes() -> float:
    return (time.time() - NOTEBOOK_START_TIME) / 60.0


def _stream_cmd(cmd, env, console_log: Path) -> int:
    in_traceback = False
    with console_log.open("a", encoding="utf-8") as log_fp:
        log_fp.write("\nCOMMAND: " + " ".join(cmd) + "\n")
        proc = subprocess.Popen(
            cmd,
            cwd=str(REPO_DIR),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            log_fp.write(line)
            if "Traceback" in line:
                in_traceback = True
            if in_traceback:
                print(line, end="", flush=True)
                continue
            if line.lstrip().startswith("Epoch: ["):
                if "  0/" in line or "100/" in line or "200/" in line:
                    print(line, end="", flush=True)
                continue
            if line.lstrip().startswith("Test:  ["):
                continue
            if any(tok in line for tok in [
                "Averaged stats:", "best_stat:", "Training time",
                "Average Precision", "number of trainable"
            ]):
                print(line, end="", flush=True)
        return proc.wait()


def run_training_experiment(
    exp_name: str,
    config_rel_path: str,
    run_out_dir: Path,
    target_epochs: int,
    milestone_epoch: int,
    master_port: int = 29501,
) -> dict:
    run_out_dir.mkdir(parents=True, exist_ok=True)

    train_img_dir = VISDRONE_ROOT / "VisDrone2019-DET-train" / "images"
    train_ann_file = VISDRONE_ROOT / "annotations" / "instances_train.json"
    val_img_dir = VISDRONE_ROOT / "VisDrone2019-DET-val" / "images"
    val_ann_file = VISDRONE_ROOT / "annotations" / "instances_val.json"

    base_updates = [
        f"train_dataloader.dataset.img_folder={train_img_dir}",
        f"train_dataloader.dataset.ann_file={train_ann_file}",
        f"val_dataloader.dataset.img_folder={val_img_dir}",
        f"val_dataloader.dataset.ann_file={val_ann_file}",
        f"epoches={target_epochs}",
        f"lr_scheduler.milestones=[{milestone_epoch}]",
        "sync_bn=False",
    ]

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["NCCL_P2P_DISABLE"] = "1"
    env["NCCL_IB_DISABLE"] = "1"
    env["OMP_NUM_THREADS"] = "2"
    env["MKL_NUM_THREADS"] = "2"

    console_log = ARTIFACTS_DIR / f"{exp_name}_console.log"
    if console_log.exists():
        console_log.unlink()

    t0 = time.time()
    ret = -1

    if NUM_GPUS >= 2:
        env_ddp = env.copy()
        env_ddp["CUDA_VISIBLE_DEVICES"] = "0,1"
        cmd_ddp = [
            "torchrun",
            "--nproc_per_node=2",
            f"--master_port={master_port}",
            "tools/train.py",
            "-c", config_rel_path,
            "-t", str(PRETRAINED_PATH),
            "--seed", "42",
            "--use-amp",
            "--output-dir", str(run_out_dir),
            "-u", *base_updates,
        ]
        print("=" * 88)
        print(f"[{exp_name.upper()}] Launching on 2 GPUs (DDP) | Full epochs: {target_epochs}")
        print("=" * 88)
        ret = _stream_cmd(cmd_ddp, env_ddp, console_log)
        if ret != 0:
            print(f"\n[WARN] 2-GPU DDP exited with code {ret}; automatically failing over to single-GPU cuda:0 ...")

    if ret != 0:
        single_updates = [
            *base_updates,
            "train_dataloader.total_batch_size=16",
            "val_dataloader.total_batch_size=16",
        ]
        last_ckpt = run_out_dir / "last.pth"
        ckpt_args = ["-r", str(last_ckpt)] if last_ckpt.exists() else ["-t", str(PRETRAINED_PATH)]
        env_single = env.copy()
        env_single["CUDA_VISIBLE_DEVICES"] = "0"
        cmd_single = [
            sys.executable,
            "tools/train.py",
            "-c", config_rel_path,
            *ckpt_args,
            "--device", "cuda:0",
            "--seed", "42",
            "--use-amp",
            "--output-dir", str(run_out_dir),
            "-u", *single_updates,
        ]
        print("=" * 88)
        print(f"[{exp_name.upper()}] Running on 1 GPU (cuda:0) | Full epochs: {target_epochs}")
        print("=" * 88)
        ret = _stream_cmd(cmd_single, env_single, console_log)

    if ret != 0:
        tail = "\n".join(console_log.read_text(encoding="utf-8").splitlines()[-60:])
        print("\n--- LAST 60 LINES OF CONSOLE LOG ---\n" + tail)
        raise RuntimeError(f"Training {exp_name} exited with code {ret}")

    wall_min = (time.time() - t0) / 60.0
    summary_file = run_out_dir / "summary.json"
    summary = json.loads(summary_file.read_text(encoding="utf-8")) if summary_file.exists() else {}
    summary["wall_minutes"] = round(wall_min, 2)
    print(f"\n[{exp_name.upper()}] Finished in {wall_min:.1f} min | Summary: {summary}\n")
    return summary

## Step 5: Train Baseline (`RT-DETR-R18` — Vanilla `HybridEncoder`)


In [5]:
BASELINE_RUN_DIR = TMP_ROOT / "runs" / "baseline"
TARGET_EPOCHS = 13
MILESTONE_EPOCH = 11

baseline_summary = run_training_experiment(
    exp_name="baseline_rtdetr_r18",
    config_rel_path="configs/rtdetr/rtdetr_r18vd_visdrone_baseline.yml",
    run_out_dir=BASELINE_RUN_DIR,
    target_epochs=TARGET_EPOCHS,
    milestone_epoch=MILESTONE_EPOCH,
    master_port=29501,
)

with (ARTIFACTS_DIR / "baseline_summary.json").open("w", encoding="utf-8") as f:
    json.dump(baseline_summary, f, indent=2)
shutil.copy2(BASELINE_RUN_DIR / "log.txt", ARTIFACTS_DIR / "baseline_log.txt")
print(f"Baseline completed all {TARGET_EPOCHS} epochs. Elapsed total: {elapsed_minutes():.1f} min.")

[BASELINE_RTDETR_R18] Launching on 2 GPUs (DDP) | Full epochs: 15
number of trainable parameters: 20094584
Epoch: [0]  [  0/202]  eta: 1:30:19  lr: 0.000000  loss: 22.3055 (22.3055)  loss_bbox: 0.8251 (0.8251)  loss_bbox_aux_0: 0.8610 (0.8610)  loss_bbox_aux_1: 0.8295 (0.8295)  loss_bbox_aux_2: 0.8650 (0.8650)  loss_bbox_dn_0: 0.4216 (0.4216)  loss_bbox_dn_1: 0.2864 (0.2864)  loss_bbox_dn_2: 0.2818 (0.2818)  loss_giou: 2.3195 (2.3195)  loss_giou_aux_0: 2.3458 (2.3458)  loss_giou_aux_1: 2.3310 (2.3310)  loss_giou_aux_2: 2.4091 (2.4091)  loss_giou_dn_0: 1.3503 (1.3503)  loss_giou_dn_1: 1.2360 (1.2360)  loss_giou_dn_2: 1.2095 (1.2095)  loss_vfl: 0.4292 (0.4292)  loss_vfl_aux_0: 0.4069 (0.4069)  loss_vfl_aux_1: 0.4404 (0.4404)  loss_vfl_aux_2: 0.2893 (0.2893)  loss_vfl_dn_0: 0.9482 (0.9482)  loss_vfl_dn_1: 1.1211 (1.1211)  loss_vfl_dn_2: 1.0984 (1.0984)  time: 26.8288  data: 2.8497  max mem: 6250
Epoch: [0]  [100/202]  eta: 0:02:33  lr: 0.000010  loss: 16.1052 (20.0149)  loss_bbox: 0.4002 

## Step 6: Train Proposed Method (`Wave-CCFF RT-DETR-R18` — `WaveHybridEncoder`)


In [6]:
WAVE_RUN_DIR = TMP_ROOT / "runs" / "wave_ccff"

wave_summary = run_training_experiment(
    exp_name="wave_ccff_rtdetr_r18",
    config_rel_path="configs/rtdetr/rtdetr_r18vd_visdrone_wave_ccff.yml",
    run_out_dir=WAVE_RUN_DIR,
    target_epochs=TARGET_EPOCHS,
    milestone_epoch=MILESTONE_EPOCH,
    master_port=29502,
)

with (ARTIFACTS_DIR / "wave_ccff_summary.json").open("w", encoding="utf-8") as f:
    json.dump(wave_summary, f, indent=2)
shutil.copy2(WAVE_RUN_DIR / "log.txt", ARTIFACTS_DIR / "wave_ccff_log.txt")
print(f"Wave-CCFF completed all {TARGET_EPOCHS} epochs. Elapsed total: {elapsed_minutes():.1f} min.")

[WAVE_CCFF_RTDETR_R18] Launching on 2 GPUs (DDP) | Full epochs: 13
number of trainable parameters: 20369784
Epoch: [0]  [  0/202]  eta: 0:44:36  lr: 0.000000  loss: 25.1339 (25.1339)  loss_bbox: 1.6438 (1.6438)  loss_bbox_aux_0: 1.8718 (1.8718)  loss_bbox_aux_1: 1.7025 (1.7025)  loss_bbox_aux_2: 1.7981 (1.7981)  loss_bbox_dn_0: 0.7882 (0.7882)  loss_bbox_dn_1: 0.2482 (0.2482)  loss_bbox_dn_2: 0.2491 (0.2491)  loss_giou: 2.3083 (2.3083)  loss_giou_aux_0: 2.3120 (2.3120)  loss_giou_aux_1: 2.3126 (2.3126)  loss_giou_aux_2: 2.3448 (2.3448)  loss_giou_dn_0: 1.4186 (1.4186)  loss_giou_dn_1: 1.2030 (1.2030)  loss_giou_dn_2: 1.1491 (1.1491)  loss_vfl: 0.1911 (0.1911)  loss_vfl_aux_0: 0.1362 (0.1362)  loss_vfl_aux_1: 0.1792 (0.1792)  loss_vfl_aux_2: 0.1051 (0.1051)  loss_vfl_dn_0: 0.7601 (0.7601)  loss_vfl_dn_1: 1.1775 (1.1775)  loss_vfl_dn_2: 1.2346 (1.2346)  time: 13.2491  data: 2.2644  max mem: 6526
Epoch: [0]  [100/202]  eta: 0:02:44  lr: 0.000010  loss: 17.1901 (21.1270)  loss_bbox: 0.4914

## Step 7: Official COCO/VisDrone Evaluation & Deployed T4 FP16 Latency Benchmark


In [7]:
import json
import subprocess
import sys


def pick_best_checkpoint(run_dir: Path) -> Path:
    best_p = run_dir / "best.pth"
    last_p = run_dir / "last.pth"
    if best_p.exists():
        return best_p
    if last_p.exists():
        return last_p
    raise FileNotFoundError(f"No checkpoint found in {run_dir}")


def export_slim_checkpoint(src_ckpt: Path, dst_ckpt: Path) -> float:
    import torch
    state = torch.load(src_ckpt, map_location="cpu", weights_only=False)
    slim = {
        "last_epoch": state.get("last_epoch", -1),
        "date": state.get("date", ""),
    }
    if "ema" in state:
        slim["ema"] = state["ema"]
    if "model" in state:
        slim["model"] = state["model"]
    torch.save(slim, dst_ckpt)
    return round(dst_ckpt.stat().st_size / (1024 * 1024), 2)


base_raw_ckpt = pick_best_checkpoint(BASELINE_RUN_DIR)
wave_raw_ckpt = pick_best_checkpoint(WAVE_RUN_DIR)

base_slim_ckpt = ARTIFACTS_DIR / "baseline_best.pth"
wave_slim_ckpt = ARTIFACTS_DIR / "wave_ccff_best.pth"
base_mb = export_slim_checkpoint(base_raw_ckpt, base_slim_ckpt)
wave_mb = export_slim_checkpoint(wave_raw_ckpt, wave_slim_ckpt)
print(f"Exported slim checkpoints: baseline_best.pth ({base_mb} MB), wave_ccff_best.pth ({wave_mb} MB)")

val_updates = [
    f"val_dataloader.dataset.img_folder={VISDRONE_ROOT / 'VisDrone2019-DET-val' / 'images'}",
    f"val_dataloader.dataset.ann_file={VISDRONE_ROOT / 'annotations' / 'instances_val.json'}",
    "val_dataloader.total_batch_size=16",
]

print("\n[Eval 1/2] Evaluating Baseline RT-DETR-R18 on VisDrone2019-DET val ...")
subprocess.run(
    [
        sys.executable, "tools/evaluate_checkpoint.py",
        "-c", "configs/rtdetr/rtdetr_r18vd_visdrone_baseline.yml",
        "-r", str(base_slim_ckpt),
        "--device", "cuda:0",
        "--out", str(ARTIFACTS_DIR / "baseline_eval.json"),
        "-u", *val_updates,
    ],
    cwd=str(REPO_DIR),
    check=True,
)

print("\n[Eval 2/2] Evaluating Proposed Wave-CCFF RT-DETR-R18 on VisDrone2019-DET val ...")
subprocess.run(
    [
        sys.executable, "tools/evaluate_checkpoint.py",
        "-c", "configs/rtdetr/rtdetr_r18vd_visdrone_wave_ccff.yml",
        "-r", str(wave_slim_ckpt),
        "--device", "cuda:0",
        "--out", str(ARTIFACTS_DIR / "wave_ccff_eval.json"),
        "-u", *val_updates,
    ],
    cwd=str(REPO_DIR),
    check=True,
)

print("\n[Latency 1/2] Benchmarking Baseline deployed FP16 latency on T4 (batch=1, 640x640) ...")
subprocess.run(
    [
        sys.executable, "tools/benchmark_latency.py",
        "-c", "configs/rtdetr/rtdetr_r18vd_visdrone_baseline.yml",
        "-r", str(base_slim_ckpt),
        "--device", "cuda:0",
        "--img-size", "640",
        "--warmup", "100",
        "--iters", "300",
        "--out", str(ARTIFACTS_DIR / "baseline_latency.json"),
    ],
    cwd=str(REPO_DIR),
    check=True,
)

print("\n[Latency 2/2] Benchmarking Wave-CCFF deployed FP16 latency on T4 (batch=1, 640x640) ...")
subprocess.run(
    [
        sys.executable, "tools/benchmark_latency.py",
        "-c", "configs/rtdetr/rtdetr_r18vd_visdrone_wave_ccff.yml",
        "-r", str(wave_slim_ckpt),
        "--device", "cuda:0",
        "--img-size", "640",
        "--warmup", "100",
        "--iters", "300",
        "--out", str(ARTIFACTS_DIR / "wave_ccff_latency.json"),
    ],
    cwd=str(REPO_DIR),
    check=True,
)

Exported slim checkpoints: baseline_best.pth (153.89 MB), wave_ccff_best.pth (156.04 MB)

[Eval 1/2] Evaluating Baseline RT-DETR-R18 on VisDrone2019-DET val ...


I0000 00:00:1795197081.358225     790 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


building val_dataloader with batch_size=16...
Test:  [ 0/35]  eta: 0:01:18    time: 2.2428  data: 0.9429  max mem: 815
Test:  [10/35]  eta: 0:00:11    time: 0.4564  data: 0.1730  max mem: 815
Test:  [20/35]  eta: 0:00:05    time: 0.2962  data: 0.1156  max mem: 815
Test:  [30/35]  eta: 0:00:01    time: 0.3373  data: 0.1241  max mem: 815
Test:  [34/35]  eta: 0:00:00    time: 0.3266  data: 0.1150  max mem: 815
Test: Total time: 0:00:12 (0.3662 s / it)
Averaged stats: 
IoU metric: bbox
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.185
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.302
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.187
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.106
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.280
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.535
 Average Recall     (AR) @[ IoU=0

I0000 00:00:1795197129.407373     817 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


building val_dataloader with batch_size=16...
Test:  [ 0/35]  eta: 0:01:19    time: 2.2637  data: 0.8953  max mem: 867
Test:  [10/35]  eta: 0:00:12    time: 0.4948  data: 0.1810  max mem: 867
Test:  [20/35]  eta: 0:00:06    time: 0.3176  data: 0.1110  max mem: 867
Test:  [30/35]  eta: 0:00:02    time: 0.3507  data: 0.1445  max mem: 867
Test:  [34/35]  eta: 0:00:00    time: 0.3385  data: 0.1329  max mem: 867
Test: Total time: 0:00:13 (0.3885 s / it)
Averaged stats: 
IoU metric: bbox
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.179
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.293
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.181
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.101
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.273
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.490
 Average Recall     (AR) @[ IoU=0

I0000 00:00:1795197177.396805     844 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


{
  "config": "configs/rtdetr/rtdetr_r18vd_visdrone_baseline.yml",
  "device": "Tesla T4",
  "fp16": true,
  "img_size": 640,
  "params_M": 20.09,
  "params_exact_M": 20.0946,
  "deployed_params_M": 21.87,
  "latency_ms": 22.35,
  "fps": 44.7
}

[Latency 2/2] Benchmarking Wave-CCFF deployed FP16 latency on T4 (batch=1, 640x640) ...


I0000 00:00:1795197199.545706     863 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


{
  "config": "configs/rtdetr/rtdetr_r18vd_visdrone_wave_ccff.yml",
  "device": "Tesla T4",
  "fp16": true,
  "img_size": 640,
  "params_M": 20.37,
  "params_exact_M": 20.3698,
  "deployed_params_M": 22.14,
  "latency_ms": 26.13,
  "fps": 38.3
}


CompletedProcess(args=['/usr/bin/python3', 'tools/benchmark_latency.py', '-c', 'configs/rtdetr/rtdetr_r18vd_visdrone_wave_ccff.yml', '-r', '/kaggle/working/submission_artifacts/wave_ccff_best.pth', '--device', 'cuda:0', '--img-size', '640', '--warmup', '100', '--iters', '300', '--out', '/kaggle/working/submission_artifacts/wave_ccff_latency.json'], returncode=0)

## Step 8: Plots, Pre-Filled Overleaf `main.tex` ZIP, & Final Deliverables


In [8]:
import csv
import json
import zipfile
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

base_eval = json.loads((ARTIFACTS_DIR / "baseline_eval.json").read_text(encoding="utf-8"))
wave_eval = json.loads((ARTIFACTS_DIR / "wave_ccff_eval.json").read_text(encoding="utf-8"))
base_lat = json.loads((ARTIFACTS_DIR / "baseline_latency.json").read_text(encoding="utf-8"))
wave_lat = json.loads((ARTIFACTS_DIR / "wave_ccff_latency.json").read_text(encoding="utf-8"))


def fmt_delta(val: float) -> str:
    sign = "+" if val >= 0 else ""
    return f"{sign}{val:.2f}"


delta_ap = round(wave_eval["AP"] - base_eval["AP"], 2)
delta_aps = round(wave_eval["APS"] - base_eval["APS"], 2)
delta_ap50 = round(wave_eval["AP50"] - base_eval["AP50"], 2)
delta_ap75 = round(wave_eval["AP75"] - base_eval["AP75"], 2)
delta_params = round(wave_lat["params_M"] - base_lat["params_M"], 2)
delta_lat = round(wave_lat["latency_ms"] - base_lat["latency_ms"], 2)

final_results = {
    "dataset": "VisDrone2019-DET (val, 548 images, 640x640)",
    "gpu": base_lat["device"],
    "baseline": {**base_eval, **base_lat},
    "wave_ccff": {**wave_eval, **wave_lat},
    "delta": {
        "AP": delta_ap,
        "APS": delta_aps,
        "AP50": delta_ap50,
        "AP75": delta_ap75,
        "params_M": delta_params,
        "latency_ms": delta_lat,
    },
    "total_runtime_minutes": round(elapsed_minutes(), 2),
}
with (ARTIFACTS_DIR / "final_results.json").open("w", encoding="utf-8") as f:
    json.dump(final_results, f, indent=2)

csv_path = ARTIFACTS_DIR / "comparison_table.csv"
with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Method", "AP", "AP50", "AP75", "APS", "APM", "APL", "Params_M", "Latency_ms", "FPS"])
    writer.writerow([
        "RT-DETR-R18 (Baseline)", base_eval["AP"], base_eval["AP50"], base_eval["AP75"],
        base_eval["APS"], base_eval["APM"], base_eval["APL"],
        base_lat["params_M"], base_lat["latency_ms"], base_lat["fps"],
    ])
    writer.writerow([
        "Wave-CCFF RT-DETR-R18 (Ours)", wave_eval["AP"], wave_eval["AP50"], wave_eval["AP75"],
        wave_eval["APS"], wave_eval["APM"], wave_eval["APL"],
        wave_lat["params_M"], wave_lat["latency_ms"], wave_lat["fps"],
    ])
    writer.writerow([
        "Delta (Ours - Baseline)", fmt_delta(delta_ap), fmt_delta(delta_ap50), fmt_delta(delta_ap75),
        fmt_delta(delta_aps), fmt_delta(wave_eval["APM"] - base_eval["APM"]),
        fmt_delta(wave_eval["APL"] - base_eval["APL"]),
        fmt_delta(delta_params), fmt_delta(delta_lat),
        fmt_delta(wave_lat["fps"] - base_lat["fps"]),
    ])

def parse_log_curves(log_path: Path):
    ep, ap, aps, loss = [], [], [], []
    for line in log_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        ep.append(row["epoch"] + 1)
        bbox = row.get("test_coco_eval_bbox", [0]*6)
        ap.append(bbox[0] * 100.0)
        aps.append(bbox[3] * 100.0)
        loss.append(row.get("train_loss", 0.0))
    return ep, ap, aps, loss

b_ep, b_ap, b_aps, b_loss = parse_log_curves(ARTIFACTS_DIR / "baseline_log.txt")
w_ep, w_ap, w_aps, w_loss = parse_log_curves(ARTIFACTS_DIR / "wave_ccff_log.txt")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), dpi=180)
axes[0].plot(b_ep, b_ap, "o--", label="Baseline RT-DETR-R18", color="#4C72B0", markersize=4)
axes[0].plot(w_ep, w_ap, "s-", label="Wave-CCFF RT-DETR-R18 (Ours)", color="#DD8452", markersize=4)
axes[0].set_title("Validation AP (IoU=0.50:0.95)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("AP (%)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(b_ep, b_aps, "o--", label="Baseline RT-DETR-R18", color="#4C72B0", markersize=4)
axes[1].plot(w_ep, w_aps, "s-", label="Wave-CCFF RT-DETR-R18 (Ours)", color="#DD8452", markersize=4)
axes[1].set_title("Validation Small-Object AP_S (area < 32^2)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("AP_S (%)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(b_ep, b_loss, "o--", label="Baseline", color="#4C72B0", markersize=4)
axes[2].plot(w_ep, w_loss, "s-", label="Wave-CCFF (Ours)", color="#DD8452", markersize=4)
axes[2].set_title("Training Loss")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Total Loss")
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "training_curves.png", bbox_inches="tight")
plt.close(fig)

val_img_folder = VISDRONE_ROOT / "VisDrone2019-DET-val" / "images"
val_imgs = sorted(val_img_folder.glob("*.jpg"))[:3] if val_img_folder.exists() else []
if val_imgs:
    vis_tmp_b = TMP_ROOT / "vis_base"
    vis_tmp_w = TMP_ROOT / "vis_wave"
    vis_sample_in = TMP_ROOT / "vis_in"
    vis_sample_in.mkdir(parents=True, exist_ok=True)
    for p in val_imgs:
        shutil.copy2(p, vis_sample_in / p.name)

    subprocess.run([
        sys.executable, "tools/infer_visualize.py",
        "-c", "configs/rtdetr/rtdetr_r18vd_visdrone_baseline.yml",
        "-r", str(base_slim_ckpt), "-i", str(vis_sample_in), "-o", str(vis_tmp_b), "--score-thr", "0.35"
    ], cwd=str(REPO_DIR), check=True)
    subprocess.run([
        sys.executable, "tools/infer_visualize.py",
        "-c", "configs/rtdetr/rtdetr_r18vd_visdrone_wave_ccff.yml",
        "-r", str(wave_slim_ckpt), "-i", str(vis_sample_in), "-o", str(vis_tmp_w), "--score-thr", "0.35"
    ], cwd=str(REPO_DIR), check=True)

    fig_q, ax_q = plt.subplots(len(val_imgs), 2, figsize=(14, 4.2 * len(val_imgs)), dpi=150)
    if len(val_imgs) == 1:
        ax_q = [ax_q]
    for row_i, p in enumerate(val_imgs):
        ax_q[row_i][0].imshow(Image.open(vis_tmp_b / p.name))
        ax_q[row_i][0].set_title(f"Baseline RT-DETR-R18 — {p.name}")
        ax_q[row_i][0].axis("off")
        ax_q[row_i][1].imshow(Image.open(vis_tmp_w / p.name))
        ax_q[row_i][1].set_title(f"Wave-CCFF RT-DETR-R18 (Ours) — {p.name}")
        ax_q[row_i][1].axis("off")
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / "qualitative_comparison.png", bbox_inches="tight")
    plt.close(fig_q)

b_ap_s = f"{base_eval['AP']:.2f}"
w_ap_s = f"{wave_eval['AP']:.2f}"
b_aps_s = f"{base_eval['APS']:.2f}"
w_aps_s = f"{wave_eval['APS']:.2f}"
b_lat_s = f"{base_lat['latency_ms']:.2f}"
w_lat_s = f"{wave_lat['latency_ms']:.2f}"

best_ap_b = f"\\textbf{{{b_ap_s}}}" if base_eval["AP"] > wave_eval["AP"] else b_ap_s
best_ap_w = f"\\textbf{{{w_ap_s}}}" if wave_eval["AP"] >= base_eval["AP"] else w_ap_s
best_aps_b = f"\\textbf{{{b_aps_s}}}" if base_eval["APS"] > wave_eval["APS"] else b_aps_s
best_aps_w = f"\\textbf{{{w_aps_s}}}" if wave_eval["APS"] >= base_eval["APS"] else w_aps_s
best_lat_b = f"\\textbf{{{b_lat_s}}}" if base_lat["latency_ms"] <= wave_lat["latency_ms"] else b_lat_s
best_lat_w = f"\\textbf{{{w_lat_s}}}" if wave_lat["latency_ms"] < base_lat["latency_ms"] else w_lat_s

latex_rows = "\n".join([
    "RT-DETR-R18 (Baseline)",
    "&",
    best_ap_b,
    "&",
    best_aps_b,
    "&",
    f"{base_lat['params_M']:.2f}",
    "&",
    best_lat_b,
    r"\\",
    "",
    r"\textbf{Wave-CCFF RT-DETR-R18 (Ours)}",
    "&",
    best_ap_w,
    "&",
    best_aps_w,
    "&",
    f"{wave_lat['params_M']:.2f}",
    "&",
    best_lat_w,
    r"\\",
    "",
    r"\midrule",
    "",
    r"$\Delta$ (Ours -- Baseline)",
    "&",
    fmt_delta(delta_ap),
    "&",
    fmt_delta(delta_aps),
    "&",
    fmt_delta(delta_params),
    "&",
    fmt_delta(delta_lat),
    r"\\",
])

(ARTIFACTS_DIR / "latex_results_snippet.tex").write_text(latex_rows, encoding="utf-8")

tex_template = (REPO_DIR / "report_template" / "main.tex").read_text(encoding="utf-8")
placeholder_block = "\n".join([
    "RT-DETR-R18 (Baseline)",
    "&",
    "XX.XX",
    "&",
    "XX.XX",
    "&",
    "20.09",
    "&",
    "X.XX",
    r"\\",
    "",
    r"\textbf{Wave-CCFF RT-DETR-R18 (Ours)}",
    "&",
    r"\textbf{XX.XX}",
    "&",
    r"\textbf{XX.XX}",
    "&",
    "20.37",
    "&",
    "X.XX",
    r"\\",
    "",
    r"\midrule",
    "",
    r"$\Delta$ (Ours -- Baseline)",
    "&",
    "+X.XX",
    "&",
    "+X.XX",
    "&",
    "+0.28",
    "&",
    "+0.XX",
    r"\\",
])

assert placeholder_block in tex_template, "Template table placeholder mismatch!"
filled_tex = tex_template.replace(placeholder_block, latex_rows)
if len(w_ep) > 0:
    filled_tex = filled_tex.replace("15~epochs", f"{len(w_ep)}~epochs")
(ARTIFACTS_DIR / "main.tex").write_text(filled_tex, encoding="utf-8")

overleaf_zip = WORK_ROOT / "Overleaf_Upload_Wave_CCFF.zip"
with zipfile.ZipFile(overleaf_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("main.tex", filled_tex)

results_zip = WORK_ROOT / "da6021_report_results_pack.zip"
with zipfile.ZipFile(results_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in sorted(ARTIFACTS_DIR.iterdir()):
        if item.suffix != ".pth":
            zf.write(item, arcname=item.name)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

print("=" * 88)
print(f"ALL DONE IN {elapsed_minutes():.1f} MINUTES!")
print("=" * 88)
print("\n--- FINAL QUANTITATIVE RESULTS (VisDrone2019-DET val, 640x640, T4 FP16) ---")
print((ARTIFACTS_DIR / "comparison_table.csv").read_text(encoding="utf-8"))
print("--- COPY-PASTE LATEX TABLE ROWS (ALSO PRE-FILLED IN Overleaf_Upload_Wave_CCFF.zip) ---")
print(latex_rows)
print("\nDownloadable Output Files in /kaggle/working/:")
print(f"  1. {overleaf_zip}  (Upload directly to Overleaf!)")
print(f"  2. {results_zip}   (Logs, JSONs, CSV, curves & visualizations)")
print(f"  3. {base_slim_ckpt}")
print(f"  4. {wave_slim_ckpt}")

I0000 00:00:1795197225.352187     882 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Saved /tmp/da6021_wave_rtdetr/vis_base/0000001_02999_d_0000005.jpg (162 detections >= 0.35)
Saved /tmp/da6021_wave_rtdetr/vis_base/0000001_03499_d_0000006.jpg (125 detections >= 0.35)
Saved /tmp/da6021_wave_rtdetr/vis_base/0000001_03999_d_0000007.jpg (139 detections >= 0.35)


I0000 00:00:1795197238.157663     901 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Saved /tmp/da6021_wave_rtdetr/vis_wave/0000001_02999_d_0000005.jpg (161 detections >= 0.35)
Saved /tmp/da6021_wave_rtdetr/vis_wave/0000001_03499_d_0000006.jpg (135 detections >= 0.35)
Saved /tmp/da6021_wave_rtdetr/vis_wave/0000001_03999_d_0000007.jpg (147 detections >= 0.35)
ALL DONE IN 122.8 MINUTES (FULL TRAINING RUN)!

--- FINAL QUANTITATIVE RESULTS (VisDrone2019-DET val, 640x640, T4 FP16) ---
Method,AP,AP50,AP75,APS,APM,APL,Params_M,Latency_ms,FPS
RT-DETR-R18 (Baseline),18.49,30.16,18.71,10.56,28.03,53.45,20.09,22.35,44.7
Wave-CCFF RT-DETR-R18 (Ours),17.92,29.31,18.08,10.13,27.35,49.05,20.37,26.13,38.3
Delta (Ours - Baseline),-0.57,-0.85,-0.63,-0.43,-0.68,-4.40,+0.28,+3.78,-6.40

--- COPY-PASTE LATEX TABLE ROWS (ALSO PRE-FILLED IN Overleaf_Upload_Wave_CCFF.zip) ---
RT-DETR-R18 (Baseline)
&
\textbf{18.49}
&
\textbf{10.56}
&
20.09
&
\textbf{22.35}
\\

\textbf{Wave-CCFF RT-DETR-R18 (Ours)}
&
17.92
&
10.13
&
20.37
&
26.13
\\

\midrule

$\Delta$ (Ours -- Baseline)
&
-0.57
&
-0.43
&
+0.2